# Model Experimentation — Asteroid Hazard Prediction

This notebook figures out which ML model works best for predicting hazardous asteroids.
The winning model and its parameters get used in model_trainer.py in the production pipeline.

Dataset: 338,199 asteroid records from NASA NeoWs (1910–2024)
Target: is_hazardous — True = potentially dangerous to Earth

Why accuracy is not our metric:
87% of asteroids are not hazardous. A model that always predicts False gets 87% accuracy.
That model is completely useless. We use F1-score and Recall on the hazardous class instead.

F1-score: balances precision and recall into one number
Recall: out of all real hazardous asteroids, how many did we catch?
Precision: out of all asteroids we flagged as hazardous, how many actually were?

Missing a real hazardous asteroid is worse than a false alarm — so Recall matters most,
but F1 keeps Precision in check so the model doesn't just flag everything as dangerous.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

sns.set_style("darkgrid")

## 1. Data Preparation

Applying all decisions from EDA:
- Drop neo_id, name, orbiting_body — identifiers or constants, zero predictive value
- Drop estimated_diameter_min — perfect duplicate of estimated_diameter_max (correlation = 1.0)
- Drop 28 rows with missing values — 0.008% of data, not worth imputing
- RobustScaler — handles skewed features and outliers better than StandardScaler
- SMOTE on train set only — synthetic oversampling to fix 87/13 class imbalance

In [ ]:
df = pd.read_csv("../data/raw.csv")

cols_to_drop = ["neo_id", "name", "orbiting_body", "estimated_diameter_min"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
df = df.dropna()

X = df.drop(columns=["is_hazardous"])
y = df["is_hazardous"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Shape after cleaning: {df.shape}")
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Train class balance:\n{y_train.value_counts()}")

## 2. Preprocessing + SMOTE

Scaling first, then SMOTE.
Reason: SMOTE creates synthetic data points by interpolating between existing ones.
If we SMOTE before scaling, the synthetic points are created in the wrong space.

fit_transform on train → learns the scale from training data only.
transform on test → applies the same scale, never learns from test data.
This prevents data leakage.

After SMOTE the train set doubles — both classes become equal in size.
Test set is never touched by SMOTE — it must reflect the real world distribution.

In [ ]:
pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
])

X_train_scaled = pipeline.fit_transform(X_train)
X_test_scaled = pipeline.transform(X_test)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f"After SMOTE — Train shape: {X_train_resampled.shape}")
print(f"Class distribution after SMOTE: {np.bincount(y_train_resampled)}")

## 3. Baseline Model Comparison

Testing 5 models with default settings + imbalance handling.
Primary metric: F1-score on class 1 (hazardous).
Secondary metric: Recall on class 1.

Models tested:
- Logistic Regression: simple linear model, good baseline
- Random Forest: ensemble of decision trees, handles non-linear patterns well
- Gradient Boosting: builds trees sequentially, each correcting the last
- XGBoost: optimized gradient boosting, industry standard
- CatBoost: gradient boosting optimized for categorical features

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, random_state=42),
    "XGBoost":             XGBClassifier(scale_pos_weight=7, random_state=42, eval_metric="logloss"),
    "CatBoost":            CatBoostClassifier(iterations=100, random_state=42, verbose=False),
}

results = []

for name, model in models.items():
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test_scaled)
    report = classification_report(y_test, y_pred, output_dict=True)
    results.append({
        "Model":         name,
        "Accuracy":      round(report["accuracy"], 4),
        "Precision (1)": round(report["1"]["precision"], 4),
        "Recall (1)":    round(report["1"]["recall"], 4),
        "F1 (1)":        round(report["1"]["f1-score"], 4),
    })

results_df = pd.DataFrame(results).set_index("Model")
print(results_df)

## 4. Results

| Model | Accuracy | Precision (1) | Recall (1) | F1 (1) |
|---|---|---|---|---|
| Logistic Regression | 0.7353 | 0.3101 | 0.8773 | 0.4583 |
| Random Forest | 0.9056 | 0.6107 | 0.7194 | 0.6606 |
| Gradient Boosting | 0.7337 | 0.3216 | 0.9789 | 0.4841 |
| XGBoost | 0.7354 | 0.3239 | 0.9870 | 0.4878 |
| CatBoost | 0.7553 | 0.3382 | 0.9588 | 0.5000 |

Random Forest is the clear winner on F1 (0.66).

The other models (GB, XGBoost, CatBoost) have Recall near 0.97-0.99 but Precision around 0.32.
This means they flag almost everything as hazardous — not genuine learning, just over-caution.
For every real hazardous asteroid they catch, they incorrectly flag 2 safe ones.

Random Forest is the only model that actually learned a meaningful decision boundary.
Precision 0.61 means when it says hazardous, it is right 61% of the time.
Recall 0.72 means it catches 72% of all real hazardous asteroids.
F1 0.66 is the best balance achieved across all five models.

In [ ]:
results_df[["Precision (1)", "Recall (1)", "F1 (1)"]].plot(
    kind="bar", figsize=(12, 6), colormap="coolwarm"
)
plt.title("Model Comparison — Hazardous Class Metrics")
plt.ylabel("Score")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 5. Hyperparameter Tuning — Random Forest

RandomizedSearchCV tests 10 random combinations of hyperparameters,
each evaluated with 3-fold cross validation = 30 total training runs.
Scoring metric: F1.

Result:
Best params: n_estimators=200, min_samples_split=5, max_features='log2', max_depth=None
Best CV F1: 0.9397 (this is the overall CV score, not the hazardous class F1)
Test set hazardous class F1: 0.66 — identical to default Random Forest

Conclusion: tuning did not improve the hazardous class F1.
The default Random Forest parameters are already well-suited to this dataset.
We use Random Forest with default n_estimators=100 in the production pipeline.

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5, 10],
    "max_features": ["sqrt", "log2"],
}

rf = RandomForestClassifier(class_weight="balanced", random_state=42)

search = RandomizedSearchCV(
    rf, param_grid, n_iter=10, scoring="f1",
    cv=3, random_state=42, n_jobs=-1, verbose=1,
)

search.fit(X_train_resampled, y_train_resampled)

print(f"Best params: {search.best_params_}")
print(f"Best CV F1: {search.best_score_:.4f}")

y_pred_tuned = search.best_estimator_.predict(X_test_scaled)
print(classification_report(y_test, y_pred_tuned, target_names=["Not Hazardous", "Hazardous"]))

## 6. Confusion Matrix — Random Forest

How to read this:
- Top-left (True Negative): correctly predicted not hazardous
- Top-right (False Positive): predicted hazardous but it wasn't — a false alarm
- Bottom-left (False Negative): predicted safe but it was hazardous — the dangerous error
- Bottom-right (True Positive): correctly predicted hazardous

We want the bottom-left number (False Negatives) to be as low as possible.
Missing a real hazardous asteroid is the worst outcome.

In [ ]:
best_model = models["Random Forest"]
y_pred_best = best_model.predict(X_test_scaled)

cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Hazardous", "Hazardous"]
)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix — Random Forest")
plt.show()

print(classification_report(y_test, y_pred_best, target_names=["Not Hazardous", "Hazardous"]))

## 7. Final Decision

Chosen model: Random Forest (n_estimators=100, class_weight='balanced')

Reason:
- Highest F1 on hazardous class: 0.66
- Only model with Precision above 0.60 — actually discriminates rather than flagging everything
- Tuning confirmed default parameters are already near-optimal for this dataset
- Gradient Boosting, XGBoost, CatBoost all had Precision ~0.32 — not genuine learning

This model and these preprocessing decisions are now locked in for model_trainer.py.